# Máquina Estatística - Trabalho I de Probabilidade e Estatística

1. Execute **uma vez** a célula **1. Preparação** (ela cria as funções simples usadas pelas demais células).
2. Depois disso, escolha **qualquer experimento (1 a 5)** e execute **somente a célula correspondente**, na ordem que preferir. Nenhum depende de você ter executado outro experimento.
3. Cada experimento apresenta suas próprias tabelas, resultados e, quando necessário, gráficos **aqui dentro** do notebook. Os CSV também são gerados na pasta `resultados_notebook/`.
4. Para recomeçar, reinicie o kernel e execute novamente a preparação. As sementes garantem a mesma sequência no mesmo ambiente.

**Requisito:** Python 3 com `matplotlib`. No computador, instale com `pip install matplotlib` se necessário. O notebook funciona sozinho; **não precisa** importar `maquina.py`.

**Integrantes:** preencher antes da entrega.

## 1. Preparação - Executar uma única vez

Definimos os cinco símbolos, os pagamentos, o sorteio de três paradas independentes e a regra da **linha central**. A mesma função `girar` é usada por todos os experimentos.

Cada rolo tem sequência circular `(A, B, C, D, E)`. Se a parada é `K`, a coluna mostra as posições `K`, `(K+1) % 5` e `(K+2) % 5`. O sorteio uniforme ocorre com `randrange(5)`.

In [ ]:
import csv
import math
import platform
import random
import statistics
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt

SIMBOLOS = "ABCDE"
PAGAMENTOS = {"A": 5, "B": 10, "C": 15, "D": 25, "E": 50}
APOSTA = 1
PASTA = Path("resultados_notebook")
PASTA.mkdir(exist_ok=True)


def girar(gerador, pagamentos=None):
    """Uma rodada com três rolos. Apenas a linha central pode premiar."""
    if pagamentos is None:
        pagamentos = PAGAMENTOS

    paradas = [gerador.randrange(5) for _ in range(3)]
    tabuleiro = [
        [SIMBOLOS[(k + linha) % 5] for k in paradas]
        for linha in range(3)
    ]
    centro = "".join(tabuleiro[1])
    premio = pagamentos[centro[0]] if len(set(centro)) == 1 else 0
    return {"tabuleiro": tabuleiro, "centro": centro,
            "premio": premio, "lucro": premio - APOSTA}


def simular(quantidade, semente):
    """Inicia o gerador uma única vez e executa todas as rodadas."""
    gerador = random.Random(semente)
    return [girar(gerador) for _ in range(quantidade)]


def salvar_csv(nome, cabecalho, linhas):
    with (PASTA / nome).open("w", newline="", encoding="utf-8") as arquivo:
        escritor = csv.writer(arquivo)
        escritor.writerow(cabecalho)
        escritor.writerows(linhas)


print("Simulador acadêmico - somente créditos fictícios")
print("Python:", platform.python_version(), "| matplotlib:", matplotlib.__version__)
print("Gerador: random.Random (Mersenne Twister)")
print("Modelo preparado. Agora você pode executar qualquer experimento separadamente.")

## 2. Visualizar uma rodada

Uma rodada tem três linhas e três colunas. **Só a segunda linha é avaliada**. O **prêmio bruto** `P` é o valor recebido; o **lucro líquido** é `X = P − 1`.

Essa célula é opcional e pode ser executada várias vezes. A semente `2026` reproduz o mesmo resultado de exemplo; altere-a para ver outra rodada.

In [ ]:
semente = 2026
sorteio = girar(random.Random(semente))
print("        Rolo 1   Rolo 2   Rolo 3")
for i, linha in enumerate(sorteio["tabuleiro"]):
    marca = "  <-- ÚNICA LINHA PREMIADA" if i == 1 else ""
    print("          ", "      ".join(linha), marca)
print("Linha central:", "-".join(sorteio["centro"]))
print("Prêmio bruto:", sorteio["premio"], "créditos")
print("Lucro líquido:", sorteio["lucro"], "créditos")

## 3. Matemática do modelo (teoria, sem sorteios)

Existem `5³ = 125` combinações centrais igualmente prováveis. Cinco pagam prêmio (`AAA`, `BBB`, `CCC`, `DDD`, `EEE`), logo a chance de prêmio é `5/125 = 4%`.

A variável `P` representa o **prêmio bruto** e a variável `X = P − 1` representa o **lucro líquido**.

| Linha central | Prêmio `P` | Lucro `X` | Probabilidade |
|---|---:|---:|---:|
| AAA | 5 | 4 | 1/125 |
| BBB | 10 | 9 | 1/125 |
| CCC | 15 | 14 | 1/125 |
| DDD | 25 | 24 | 1/125 |
| EEE | 50 | 49 | 1/125 |
| Outras 120 | 0 | -1 | 120/125 |

Então `E[P] = (5+10+15+25+50)/125 = 0,84`, `E[X] = -0,16` e `RTP = 84%`. O RTP **não** é a probabilidade de ganhar.

In [ ]:
combinacoes = 5 ** 3
chance_premio = 5 / combinacoes
premio_esperado = sum(PAGAMENTOS.values()) / combinacoes
lucro_esperado = premio_esperado - APOSTA
rtp = premio_esperado / APOSTA

print(f"Resultados centrais possíveis: {combinacoes}")
print(f"Probabilidade de prêmio: {chance_premio:.2%}")
print(f"Prêmio esperado: {premio_esperado:.2f} crédito")
print(f"Lucro esperado: {lucro_esperado:.2f} crédito por rodada")
print(f"RTP teórico: {rtp:.1%}")
assert combinacoes == 125 and math.isclose(lucro_esperado, -0.16)

## Experimento 1 - As frequências acompanham as probabilidades?

**Enunciado:** simular **10.000 rodadas**, usando a **semente 2026**, sem saldo. Contar as seis categorias de resultado, comparar teoria e observação e calcular média, RTP, taxa de prêmio e **desvio-padrão amostral** (divisor `N − 1`).

**Execute somente a célula abaixo** depois da preparação. Ela já produz tabela, gráfico e CSV.

In [ ]:
from collections import Counter

rodadas = simular(10000, 2026)
contagens = Counter(r["centro"] if r["premio"] else "sem prêmio" for r in rodadas)
categorias = ["AAA", "BBB", "CCC", "DDD", "EEE", "sem prêmio"]
prob_teorica = [1 / 125] * 5 + [120 / 125]
frequencias = [contagens[c] / len(rodadas) for c in categorias]
lucros = [r["lucro"] for r in rodadas]
premios = [r["premio"] for r in rodadas]

print("Categoria       Lucro   Teoria     Contagem   Freq. observada")
linhas = []
for i, nome in enumerate(categorias):
    valor_lucro = PAGAMENTOS[nome[0]] - 1 if nome != "sem prêmio" else -1
    print(f"{nome:14} {valor_lucro:>5}   {prob_teorica[i]:>7.2%}   {contagens[nome]:>8}   {frequencias[i]:>8.2%}")
    linhas.append([nome, valor_lucro, prob_teorica[i], contagens[nome], frequencias[i]])
salvar_csv("experimento1.csv", ["categoria", "lucro", "probabilidade", "contagem", "frequencia"], linhas)

print(f"\nTotal de rodadas: {sum(contagens.values())}")
print(f"Lucro médio: {statistics.mean(lucros):.4f} crédito (teoria: -0,1600)")
print(f"Desvio-padrão amostral: {statistics.stdev(lucros):.4f} créditos")
print(f"RTP observado: {statistics.mean(premios):.2%} (teoria: 84%)")
print(f"Rodadas premiadas: {sum(p > 0 for p in premios) / 10000:.2%} (teoria: 4%)")
assert sum(contagens.values()) == 10000

posicoes = range(6)
plt.figure(figsize=(9, 4))
plt.bar([p - 0.2 for p in posicoes], [100 * p for p in prob_teorica], width=0.4, label="Teoria")
plt.bar([p + 0.2 for p in posicoes], [100 * f for f in frequencias], width=0.4, label="Simulação")
plt.xticks(list(posicoes), categorias)
plt.ylabel("Probabilidade / frequência (%)")
plt.xlabel("Resultado da linha central")
plt.title("Experimento 1 — Teoria x simulação (10.000 rodadas)")
plt.legend()
plt.tight_layout()
plt.show()

**Interpretação:** as frequências se aproximam das probabilidades, mas não precisam coincidir exatamente. A chance de **algum prêmio** é aproximadamente 4%; o **RTP observado** é a soma dos prêmios dividida pelo total apostado. O lucro médio flutua ao redor de −0,16 crédito. O desvio-padrão descreve a dispersão dos lucros por rodada.

## Experimento 2 - O que muda quando aumentamos a amostra?

**Enunciado:** comparar **três sequências de 10.000 rodadas**, com sementes **10**, **2026** e **123456**. Para cada sequência, anotar a média acumulada após 100, 1.000 e 10.000 rodadas e desenhar as curvas com referência em **−0,16**.

A sequência da semente `2026` é **idêntica à do Experimento 1**, mesmo que você execute esta célula primeiro: recriamos a mesma sequência determinística. Também conferimos seus 10 primeiros resultados.

In [ ]:
sementes = [10, 2026, 123456]
pontos = [100, 1000, 10000]
linhas = []
plt.figure(figsize=(9, 4.5))

for semente in sementes:
    rodadas = simular(10000, semente)
    soma = 0
    medias = []
    for i, r in enumerate(rodadas, start=1):
        soma += r["lucro"]
        media_atual = soma / i
        medias.append(media_atual)
        if i in pontos:
            linhas.append([semente, i, media_atual])
    plt.plot(range(1, 10001, 20), medias[::20], label=f"Semente {semente}", linewidth=1.2)

print("Semente      100 rodadas    1.000 rodadas    10.000 rodadas")
for semente in sementes:
    medias_semente = [linha[2] for linha in linhas if linha[0] == semente]
    print(f"{semente:<12}" + " ".join(f"{m:>14.4f}" for m in medias_semente))
salvar_csv("experimento2.csv", ["semente", "rodadas", "lucro_medio_acumulado"], linhas)

primeiros = [r["centro"] for r in simular(10000, 2026)[:10]]
replicados = [r["centro"] for r in simular(10, 2026)]
print("\nPrimeiras 10 combinações:", primeiros)
print("Mesma semente reproduziu os 10 resultados?", primeiros == replicados)
assert primeiros == replicados

plt.axhline(-0.16, linestyle="--", color="black", label="Esperança: -0,16")
plt.xlabel("Quantidade de rodadas")
plt.ylabel("Lucro médio acumulado (créditos)")
plt.title("Experimento 2 — Lei dos Grandes Números")
plt.legend()
plt.tight_layout()
plt.show()

**Interpretação:** a Lei dos Grandes Números diz que médias de amostras grandes tendem ao valor esperado sob as hipóteses do modelo. **Não exige** que a aproximação seja sempre crescente ou que uma sequência “compense” resultados anteriores. Diferentes sementes produzem trajetórias diferentes.

## Experimento 3 - O que acontece com um saldo limitado?

**Enunciado:** 100 participantes **virtuais**, saldo inicial de **100 créditos**, aposta de **1**, no máximo **500 rodadas** por participante ou parada quando o saldo ficar abaixo de 1. Usar **semente 42 uma única vez**, sem reinicializar o gerador entre pessoas.

A célula imprime as proporções de ganho, empate, perda e esgotamento, além da média e mediana dos saldos finais. O gráfico mostra os **cinco primeiros participantes**, sem selecioná-los por resultado.

In [ ]:
gerador = random.Random(42)   # O mesmo gerador para todos os participantes
sessoes = []
trajetorias = []

for pessoa in range(1, 101):
    saldo = 100
    caminho = [saldo]
    quantidade = 0

    while saldo >= 1 and quantidade < 500:
        resultado = girar(gerador)
        saldo += resultado["lucro"]
        quantidade += 1
        caminho.append(saldo)

    situacao = "ganho" if saldo > 100 else "empate" if saldo == 100 else "perda"
    sessoes.append([pessoa, saldo, quantidade, situacao, saldo < 1])
    if pessoa <= 5:
        trajetorias.append(caminho + [saldo] * (501 - len(caminho)))

salvar_csv("experimento3.csv", ["participante", "saldo_final", "rodadas", "situacao", "esgotou"], sessoes)
ganhos = sum(s[3] == "ganho" for s in sessoes)
empates = sum(s[3] == "empate" for s in sessoes)
perdas = sum(s[3] == "perda" for s in sessoes)
esgotaram = sum(s[4] for s in sessoes)
saldos = [s[1] for s in sessoes]

print(f"Ganhos: {ganhos}%  |  Empates: {empates}%  |  Perdas: {perdas}%")
print(f"Esgotamento (dentro das perdas): {esgotaram}%")
print(f"Saldo médio: {statistics.mean(saldos):.2f} créditos")
print(f"Saldo mediano: {statistics.median(saldos):.2f} créditos")
print(f"Rodadas médias: {statistics.mean([s[2] for s in sessoes]):.1f}")
assert ganhos + empates + perdas == 100

plt.figure(figsize=(9, 4.5))
for pessoa, caminho in enumerate(trajetorias, start=1):
    plt.plot(range(501), caminho, label=f"Participante {pessoa}")
plt.axhline(100, linestyle="--", color="black", label="Saldo inicial")
plt.xlabel("Número de rodadas")
plt.ylabel("Saldo (créditos fictícios)")
plt.title("Experimento 3 — Cinco primeiros participantes")
plt.legend()
plt.tight_layout()
plt.show()

**Interpretação:** algumas sessões podem terminar com lucro, apesar da esperança negativa por rodada. A **média** pode ser afetada por poucos resultados elevados, enquanto a **mediana** descreve o participante central. O esgotamento depende também do limite de saldo e do número variável de rodadas; não é uma estimativa para famílias brasileiras.

## Experimento 4 - Depois de perder, aumenta a chance de ganhar?

**Enunciado:** usar as **10.000 rodadas da semente 2026** e analisar os **9.999 pares consecutivos**. Para cada grupo, após **rodada sem prêmio** ou após **rodada premiada**, dividir a quantidade de prêmios seguintes pelo total de pares naquele grupo.

**Hipótese do modelo:** ambas as probabilidades condicionais são 4%. Uma frequência observada diferente não significa mudança nas regras.

In [ ]:
rodadas = simular(10000, 2026)  # Mesma sequência do Experimento 1
grupos = {False: [0, 0], True: [0, 0]}  # [total de pares, prêmios seguintes]

for anterior, proxima in zip(rodadas, rodadas[1:]):
    houve_premio = anterior["premio"] > 0
    grupos[houve_premio][0] += 1
    if proxima["premio"] > 0:
        grupos[houve_premio][1] += 1

print("Resultado anterior     Pares     Prêmios seguintes    Proporção")
linhas = []
for houve_premio, nome in [(False, "sem prêmio"), (True, "com prêmio")]:
    total, premiados = grupos[houve_premio]
    proporcao = premiados / total if total else None
    descricao = f"{proporcao:.2%}" if proporcao is not None else "não estimável"
    print(f"{nome:20} {total:>5}     {premiados:>10}           {descricao}")
    linhas.append([nome, total, premiados, proporcao if proporcao is not None else "não estimável"])

salvar_csv("experimento4.csv", ["resultado_anterior", "total_pares", "premios_seguintes", "proporcao"], linhas)
print("Pares analisados:", sum(v[0] for v in grupos.values()))
print("Probabilidade teórica nos dois grupos: 4%")
assert sum(v[0] for v in grupos.values()) == 9999

**Interpretação:** rodadas independentes não têm “memória”. Depois de perder, a máquina não fica devendo um prêmio. A expectativa de compensação imediata é chamada **falácia do jogador**. O grupo “após prêmio” normalmente tem menos pares, então sua frequência pode oscilar mais.

## Experimento 5 - Prêmios maiores tornam o jogo favorável?

**Enunciado:** aplicar **duas tabelas de pagamento aos mesmos 10.000 sorteios de 2026**, sem mudar as probabilidades. A segunda aumenta cada pagamento em **10%** (ex.: `AAA = 5,5`).

Valores teóricos: `RTP_base = 84%`, `RTP_+10% = 92,4%`, `E[X_+10%] = −0,076`. A chance de prêmio permanece 4%.

In [ ]:
rodadas = simular(10000, 2026)  # Os mesmos sorteios nas duas tabelas
pagamentos_10 = {s: valor * 1.10 for s, valor in PAGAMENTOS.items()}
linhas = []

for nome, tabela in [("Base", PAGAMENTOS), ("Prêmios +10%", pagamentos_10)]:
    premios = []
    for r in rodadas:
        centro = r["centro"]
        premio = tabela[centro[0]] if len(set(centro)) == 1 else 0
        premios.append(premio)
    lucro_medio = statistics.mean(premios) - APOSTA
    premio_medio = statistics.mean(premios)
    rtp_teorico = sum(tabela.values()) / 125
    freq_premio = sum(p > 0 for p in premios) / len(premios)
    linhas.append([nome, 0.04, freq_premio, premio_medio, lucro_medio, rtp_teorico, premio_medio])

print("Tabela            Chance (teoria)   Freq. obs.   Prêmio médio   Lucro médio   RTP teórico  RTP observado")
for nome, chance, freq, premio, lucro, rtp, rtp_obs in linhas:
    print(f"{nome:16} {chance:>7.1%}          {freq:>7.2%}      {premio:>7.4f}       {lucro:>8.4f}    {rtp:>8.1%}      {rtp_obs:>8.2%}")
salvar_csv("experimento5.csv", ["tabela", "probabilidade_teorica", "frequencia_observada", "premio_medio", "lucro_medio", "rtp_teorico", "rtp_observado"], linhas)

print("\nLucro esperado +10%:", round(sum(pagamentos_10.values()) / 125 - 1, 3))
assert math.isclose(linhas[1][5], 0.924)

plt.figure(figsize=(7, 4))
plt.bar(["Base", "+10%"], [linha[5] * 100 for linha in linhas], label="RTP teórico")
plt.axhline(100, linestyle="--", color="black", label="Ponto de equilíbrio")
plt.ylabel("Retorno ao jogador (%)")
plt.title("Experimento 5 — RTP das tabelas de prêmio")
plt.legend()
plt.tight_layout()
plt.show()

**Interpretação:** pagar mais não é o mesmo que pagar **mais frequentemente**. O aumento de 10% melhora o RTP (84% → 92,4%), mas o **lucro esperado permanece negativo**. A frequência de prêmio continua em 4%, pois as probabilidades não foram alteradas.

## Extras opcionais - intervalo de confiança e sensibilidade

Estes blocos são **opcionais** e **independentes dos cinco experimentos** (basta ter executado a Preparação). Os extras completos também estão no programa `maquina.py` e no relatório do repositório.

Um intervalo aproximado de 95% para a média usa `média ± 1,96 × s / √n`. A interpretação refere-se à **incerteza amostral** sob independência e condições adequadas para a aproximação normal, não a uma garantia de resultado em sessões individuais.

In [ ]:
rodadas = simular(10000, 2026)
lucros = [r["lucro"] for r in rodadas]
media = statistics.mean(lucros)
desvio = statistics.stdev(lucros)
margem = 1.96 * desvio / math.sqrt(len(lucros))
print(f"Média: {media:.4f} crédito")
print(f"Intervalo aproximado de 95%: [{media - margem:.4f}, {media + margem:.4f}]")
print("Hipóteses: rodadas independentes e aproximação normal da média (amostra grande).")

### Sensibilidade dos pagamentos

Mantemos os mesmos símbolos e chances, e variamos apenas o valor dos prêmios. A conta é teórica: `RTP = soma dos cinco pagamentos / 125`.

In [ ]:
fatores = [0.70, 0.80, 0.90, 1.00, 1.10, 1.20, 1.30]
print("Pagamentos (% da base)   RTP teórico   Lucro esperado")
for fator in fatores:
    rtp_teorico = sum(PAGAMENTOS.values()) * fator / 125
    print(f"{fator:>10.0%}               {rtp_teorico:>7.1%}        {rtp_teorico - 1:>7.3f}")

## Conferência e limites do estudo

Ao concluir, confira: cinco triplas premiadas, prêmio zero nas demais, aposta de 1 crédito, **10.000** contagens no E1, **100** participantes no E3, **9.999** pares no E4 e as duas tabelas aplicadas aos **mesmos sorteios** no E5.

As simulações ilustram um **modelo didático**. Não demonstram a prevalência de transtornos do jogo, nem perdas reais de famílias. A discussão social e as fontes estão no relatório `relatorio/relatorio.pdf`.

**Transparência de IA:** ferramentas de IA apoiaram a organização deste notebook, explicações, programação e revisão. Os cálculos e testes foram executados e são reproduzíveis; a equipe deve entender o código e conferir os resultados.